# minilm: fine-tune + SciNUP eval (mirrors bge-final.ipynb)

Same pipeline as bge-final, base model swapped to `sentence-transformers/all-MiniLM-L6-v2`.
GPU + Internet ON. Attach the triplets dataset first.


In [ ]:
!pip install -q "sentence-transformers[train]>=5.1,<6" rank-bm25 matplotlib scikit-learn 2>&1 | tail -1
import torch
print('cuda:', torch.cuda.is_available(), '| gpus:', torch.cuda.device_count())


In [ ]:
import os
ON_KAGGLE = os.path.exists('/kaggle/working')
WORK = '/kaggle/working' if ON_KAGGLE else './outputs'
TRIPLETS = None  # set below
import glob
cands = sorted(glob.glob('/kaggle/input/**/hard_negative_triplets.csv', recursive=True))
cands += sorted(glob.glob('./data/**/hard_negative_triplets.csv', recursive=True))
cands += sorted(glob.glob('**/triplets.csv', recursive=True))
assert cands, 'Attach a triplets CSV (Kaggle dataset input) first'
TRIPLETS = cands[0]
print('using:', TRIPLETS)


In [ ]:
BASE = 'sentence-transformers/all-MiniLM-L6-v2'
OUT = f'{WORK}/minilm-finetuned-citations'
EPOCHS, BS, LR, SEQ, LIMIT = 3, 32, 2e-5, 192, 15000
import pandas as pd
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
cols = ['anchor_text', 'positive_text', 'negative_text']
df = pd.read_csv(TRIPLETS, usecols=cols).dropna()
df = df[df[cols].ne('').all(axis=1)].drop_duplicates()
anchors = df['anchor_text'].drop_duplicates().tolist()
tr_a, te_a = train_test_split(anchors, test_size=0.2, random_state=42)
tr = df[df['anchor_text'].isin(set(tr_a))].reset_index(drop=True)
te = df[df['anchor_text'].isin(set(te_a))].reset_index(drop=True)
print(f'triplets={len(df)} train={len(tr)} heldout={len(te)}')
mk = lambda d: [InputExample(texts=[r.anchor_text, r.positive_text, r.negative_text]) for r in d.itertuples()]
model = SentenceTransformer(BASE)
model.max_seq_length = SEQ
loss = losses.MultipleNegativesRankingLoss(model=model, scale=20.0)
model.fit(train_objectives=[(DataLoader(mk(tr), batch_size=BS, shuffle=True), loss)],
          epochs=EPOCHS, warmup_steps=50, optimizer_params={'lr': LR}, show_progress_bar=True)
model.save(OUT)
print('saved:', OUT)
te.to_csv(f'{WORK}/heldout.csv', index=False)


In [ ]:
# Proof: held-out triplet accuracy + ROC-AUC
import numpy as np
from sklearn.metrics import roc_auc_score
E = model.encode
A = E(te['anchor_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
P = E(te['positive_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
N = E(te['negative_text'].tolist(), convert_to_numpy=True, show_progress_bar=False)
for V in (A, P, N):
    V /= (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
acc = float(((A * P).sum(1) > (A * N).sum(1)).mean())
y_true = np.array([1] * len(te) + [0] * len(te))
y_score = np.concatenate([(A * P).sum(1), (A * N).sum(1)])
print(f'held-out triplet accuracy = {acc:.4f} (n={len(te)}) | ROC-AUC = {roc_auc_score(y_true, y_score):.4f} (random = 0.50)')


## SciNUP retrieval eval (same harness as bge-final)


In [ ]:
# Eval data: SciNUP profiles + candidates + ground-truth qrels
import os
os.makedirs('/kaggle/working/scinup', exist_ok=True)
!cd /kaggle/working/scinup && [ -f dataset.jsonl ] || (wget -q https://iai.group/downloads/SciNUP/dataset.tgz && tar xzf dataset.tgz && rm dataset.tgz sampled_users.tgz)
!cd /kaggle/working/scinup && [ -f qrels.trec ] || wget -q -O qrels.trec https://raw.githubusercontent.com/iai-group/SciNUP/main/data/retrieval_results/ground_truth_qrels.trec
!ls -la /kaggle/working/scinup/


In [ ]:
import json
import numpy as np
from collections import defaultdict

def doc_text(c):
    return f"Title: {c['title'].strip()} Abstract: {c['abstract'].strip()}"

def load_records(path, limit=0):
    recs = [json.loads(l) for l in open(path, encoding='utf-8')]
    return recs[:limit] if limit else recs

def dense_run(records, model, top_k=100, bs=64):
    runs = {}
    for i, rec in enumerate(records):
        cands = rec['candidate_items']
        ids = [c['article_id'] for c in cands]
        V = model.encode([doc_text(c) for c in cands], batch_size=bs,
                         normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
        q = model.encode([rec['nl_profile']], normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=False)[0]
        runs[rec['author_id']] = list(zip(ids, map(float, V @ q)))
        if (i + 1) % 100 == 0:
            print(f"dense {i + 1}/{len(records)}", flush=True)
    return runs

def bm25_run(records):
    from rank_bm25 import BM25Okapi
    runs = {}
    for i, rec in enumerate(records):
        ids = [c['article_id'] for c in rec['candidate_items']]
        corpus = [doc_text(c).lower().split() for c in rec['candidate_items']]
        s = BM25Okapi(corpus).get_scores(rec['nl_profile'].lower().split())
        runs[rec['author_id']] = list(zip(ids, map(float, s)))
        if (i + 1) % 200 == 0:
            print(f"bm25 {i + 1}/{len(records)}", flush=True)
    return runs

def rrf(lists, k=60):
    out = {}
    for aid in lists[0]:
        sc = defaultdict(float)
        for runs in lists:
            items = runs.get(aid, []); items = sorted(items.items(), key=lambda x: x[1], reverse=True) if isinstance(items, dict) else sorted(items, key=lambda x: x[1], reverse=True);
            for r, (doc, _) in enumerate(items, 1):
                sc[doc] += 1.0 / (k + r)
        out[aid] = sorted(sc.items(), key=lambda x: x[1], reverse=True)
    return out

def read_trec(path, qrel):
    d = defaultdict(dict)
    for line in open(path, encoding='utf-8'):
        p = line.split()
        if len(p) < 4:
            continue
        d[p[0]][p[2]] = float(p[3]) if qrel else float(p[4])
    return d

def trec_metrics(run, qrels, k=100):
    R = M = RR = ND = n = 0.0
    cnt = 0
    for qid, gold in qrels.items():
        pos = {d for d, r in gold.items() if r > 0}
        if not pos:
            continue
        cnt += 1
        order = [d for d, _ in sorted(run.get(qid, {}).items(), key=lambda x: x[1], reverse=True)]
        hits = ap = first = 0
        for i, d in enumerate(order, 1):
            if d in pos:
                hits += 1
                if not first:
                    first = i
                ap += hits / i
        M += ap / len(pos)
        RR += 1.0 / first if first else 0.0
        R += sum(1 for d in order[:k] if d in pos) / len(pos)
        dcg = sum(1.0 / (2 ** r - 1) for r, d in enumerate(order[:10], 1) if d in pos)
        ideal = sum(1.0 / (2 ** r - 1) for r in range(1, min(10, len(pos)) + 1))
        ND += dcg / ideal if ideal else 0.0
    rec = R / cnt
    return {'n': cnt, 'R@100': rec, 'MAP': M / cnt, 'MRR': RR / cnt, 'NDCG@10': ND / cnt}

def write_trec(path, runs, name, top_k=100):
    with open(path, 'w') as f:
        for aid, items in runs.items():
            seen, rank = set(), 1
            for doc, s in sorted(items, key=lambda x: x[1], reverse=True)[:top_k]:
                if doc in seen:
                    continue
                seen.add(doc)
                f.write(f'{aid} Q0 {doc} {rank} {s:.4f} {name}' + chr(10))
                rank += 1
print('eval helpers ok')


In [ ]:
from sentence_transformers import SentenceTransformer
records = load_records(str(SCINUP / 'dataset.jsonl'))
print('authors:', len(records))
ftx = SentenceTransformer(OUT)
ftx.max_seq_length = SEQ
runs_ft = dense_run(records, ftx, top_k=100, bs=64)
write_trec(f'{WORK}/minilm_ft.trec', runs_ft, 'minilm_ft')
print('wrote minilm_ft.trec')


In [ ]:
runs_bm25 = bm25_run(records)
runs_rrf = rrf([runs_ft, runs_bm25])
qrels = read_trec(str(SCINUP / 'qrels.trec'), True)
print(f"{'run':<10}{'R@100':>8}{'MAP':>8}{'MRR':>8}{'NDCG@10':>9}")
for name, r in [('minilm_ft', runs_ft), ('bm25', runs_bm25), ('rrf', runs_rrf)]:
    m = trec_metrics({a: dict(v) for a, v in r.items()}, qrels)
    print(f"{name:<10}{m['R@100']:>8.4f}{m['MAP']:>8.4f}{m['MRR']:>8.4f}{m['NDCG@10']:>9.4f}  (n={m['n']})")
print('paper ensemble: R@100=0.4136 MAP=0.2163 MRR=0.6333 NDCG@10=0.4481')
